# Voice-Commerce Agentic RAG — End-to-End Colab Demo

Runs the currently-implemented half of the project **entirely inside this notebook**, no local setup required:

1. Clone the repo
2. Install dependencies
3. Build the private-catalog vector index (`rag.search`'s data)
4. Run the offline test suite as a sanity check
5. Call `rag.search` directly
6. Stand up the **combined two-tool MCP server** (`rag.search` + `web.search`) and call both tools through the real MCP tool-discovery/tool-call path
7. Reconcile private + live results (price/availability discrepancy flagging)

**Scope note:** this covers the RAG pipeline `src/` (Shane) and `web-search-mcp/` (Clark) — the Router/Planner/Answerer-Critic LangGraph nodes, ASR/TTS, and the Streamlit UI are the rest of the team's parts and aren't reproduced here.

**`web.search` note:** without live provider API keys (Exa/Brave/Tavily/Gemini/Linkup/Newsdata), this notebook still runs end-to-end — the tool degrades gracefully to zero results (by design, see step 6b) rather than failing, and step 7 falls back to a clearly-labeled simulated live result so the reconciliation logic is still demonstrated meaningfully. Paste any one key in step 2b for a real live call.

## 1. Clone the repository

In [ ]:
import os

REPO_URL = "https://github.com/gr3enarr0w/ADSP_32028_Final.git"
REPO_DIR = "ADSP_32028_Final"

if not os.path.isdir(REPO_DIR):
    !git clone -q {REPO_URL}
else:
    print(f"{REPO_DIR}/ already present — pulling latest instead of re-cloning")
    !cd {REPO_DIR} && git pull -q

%cd {REPO_DIR}
REPO_ROOT = os.getcwd()
print("repo root:", REPO_ROOT)

## 2. Install dependencies

Installs both subprojects' pinned requirements. `sentence-transformers`/`torch` (pulled in by `requirements-rag.txt`) aren't strictly needed for this run — we use the offline, deterministic **hash embedder** (§3) so the notebook doesn't depend on downloading an embedding model — but they're installed anyway for fidelity with the documented setup in `README_shane.md`.

In [ ]:
!pip install -q -r requirements-rag.txt
!pip install -q -r web-search-mcp/requirements.txt

### 2b. (Optional) live `web.search` provider API key

Paste **one** key to see a real live web result reconciled against the private catalog in §7. Leave blank to skip — everything else still runs.

Uses Colab's `google.colab.userdata` secrets store when available (Colab's 🔑 icon in the left sidebar), falling back to a plain masked prompt otherwise.

In [ ]:
from getpass import getpass

def _get_secret(name: str) -> str:
    try:
        from google.colab import userdata
        val = userdata.get(name)
        if val:
            return val
    except Exception:
        pass
    return ""

# Try Colab secrets first (silent); only prompt interactively if none are set.
PROVIDER_KEYS = {
    "BRAVE_SEARCH_API_KEY": _get_secret("BRAVE_SEARCH_API_KEY"),
    "TAVILY_API_KEY": _get_secret("TAVILY_API_KEY"),
}
if not any(PROVIDER_KEYS.values()):
    try:
        entered = getpass("Brave Search API key (or Enter to skip): ")
    except Exception:
        # No interactive stdin available (e.g. a non-interactive "run all"
        # outside Colab's own frontend) — skip rather than block.
        entered = ""
    if entered:
        PROVIDER_KEYS["BRAVE_SEARCH_API_KEY"] = entered

for k, v in PROVIDER_KEYS.items():
    if v:
        os.environ[k] = v

HAVE_LIVE_KEY = any(PROVIDER_KEYS.values())
print("live web.search key provided:", HAVE_LIVE_KEY)

## 3. Configure the RAG pipeline

`src/rag/config.py` reads everything from environment variables (model-agnostic by design — see `.env.example`). We set these directly with `os.environ` rather than writing a `.env` file: `config.py`'s data-path defaults already resolve to absolute paths anchored on its own file location (not on the notebook's CWD), which sidesteps a relative-path footgun we hit during local testing (documented in `web-search-mcp/combined_mcp_server.py`'s comments) — so the fewer paths we set explicitly, the more robust this cell is to *where* later cells `cd`.

**`EMBEDDING_PROVIDER=hash`** is an offline, deterministic embedder built for exactly this (tests/CI) — swap to `local` (sentence-transformers, no API key, downloads a small model) for closer-to-production embedding quality; both are config-only swaps, no code changes.

In [ ]:
os.environ["EMBEDDING_PROVIDER"] = "hash"   # swap to "local" for real embeddings
os.environ["HASH_DIM"] = "384"
os.environ["USE_RERANKER"] = "false"          # reranker needs sentence-transformers' cross-encoder
print("EMBEDDING_PROVIDER =", os.environ["EMBEDDING_PROVIDER"])

## 4. Build the vector index
Ingests the shipped 24-row sample slice and builds the Qdrant (embedded, local-mode) index — the same script documented in `README_shane.md`.

In [ ]:
!bash scripts/build_index.sh

## 5. Sanity check: run the offline test suite
16 tests covering ingestion, retrieval/filters, the MCP tool body, and the reconciliation logic (`tests/`) — all offline, no network calls.

In [ ]:
!PYTHONPATH=src python -m pytest tests/ -q

## 6. `rag.search` — direct call
Private-catalog hybrid retrieval (vector + BM25 + metadata filters), called directly as a plain Python function first, before going through the MCP layer in the next section.

In [ ]:
import sys
sys.path.insert(0, os.path.join(REPO_ROOT, "src"))

from rag.rag_search import rag_search
import pandas as pd

rag_out = rag_search(
    query="eco-friendly stainless steel cleaner under $15",
    k=3,
    filters={"price_max": 15, "material": "stainless steel"},
)
print(f"{rag_out['count']} hits for {rag_out['query']!r}")
pd.DataFrame(rag_out["results"])[["sku", "title", "brand", "price", "rating", "doc_id"]]

## 7. The combined two-tool MCP server

`web-search-mcp/combined_mcp_server.py` registers **both** `rag.search` (Shane — imported unchanged from `src/rag/rag_search.py`) and `web.search` (Clark — thin wrapper over a multi-provider research orchestrator) on one `FastMCP` server instance. Below we import the module in-process and drive it through FastMCP's own tool-manager, exercising the real tool-discovery + tool-call path (as a real MCP client over stdio would) rather than calling the underlying functions directly.

In [ ]:
sys.path.insert(0, os.path.join(REPO_ROOT, "web-search-mcp"))
import combined_mcp_server as mcp_server

tools = await mcp_server.mcp.list_tools()
for t in tools:
    print(f"- {t.name}: {t.description.splitlines()[0]}")

### 7a. `rag.search` via the MCP tool-call path
Watch the log line printed to stderr below the output — timestamped request/response logging, per the MCP-logging grading criterion.

In [ ]:
result = await mcp_server.mcp.call_tool(
    "rag.search", {"query": "plant-based dish soap gentle on hands", "k": 3}
)
import json

def _tool_json(call_result):
    """FastMCP's call_tool() returns a list of content blocks; unwrap the
    first text block's JSON payload."""
    return json.loads(call_result[0].text)

rag_via_mcp = _tool_json(result)
pd.DataFrame(rag_via_mcp["results"])[["sku", "title", "brand", "price", "rating"]]

### 7b. `web.search` via the MCP tool-call path
With no key provided (§2b), this degrades gracefully to `count: 0` — by design, not a crash (the underlying multi-provider orchestrator warns and moves on when a provider has no API key configured). With a key, it returns real live results, each carrying a `url` citation.

In [ ]:
result = await mcp_server.mcp.call_tool(
    "web.search",
    {"query": "current price of OXO Good Grips stainless steel cleaner", "k": 3},
)
web_via_mcp = _tool_json(result)
print(f"{web_via_mcp['count']} live hits" + ("" if HAVE_LIVE_KEY else " (expected: 0, no provider key was set in §2b)"))
pd.DataFrame(web_via_mcp["results"]) if web_via_mcp["results"] else web_via_mcp

## 8. Reconciliation — merging private + live results

Per `prompts/retriever_tool_instructions.md`: match private (`rag.search`) ↔ live (`web.search`) items by `sku`, then `brand`, then fuzzy `title`; the private fact always stays the grounded baseline, and price differences over 10% (or a live-only availability signal) are flagged with both citations rather than silently overwritten.

If no live results came back in §7b (no API key), we fall back to one **clearly-labeled simulated** live result — matched against the top hit from §6 — purely so the reconciliation/discrepancy-flagging logic is still visibly exercised end-to-end.

In [ ]:
from rag.reconcile import reconcile

web_results_for_reconcile = web_via_mcp["results"]
simulated = False
if not web_results_for_reconcile:
    simulated = True
    top = rag_out["results"][0]
    web_results_for_reconcile = [{
        "title": top["title"],
        "url": "https://retailer.example.com/simulated-listing",
        "snippet": "Simulated live listing for demo purposes (no provider API key set).",
        "price": round(top["price"] * 1.2, 2),  # +20% -> should trip the >10% discrepancy flag
        "availability": "in_stock",
    }]

print("live results are", "SIMULATED (no API key)" if simulated else "REAL")
reconciled = reconcile(rag_out["results"], web_results_for_reconcile)

rows = []
for item in reconciled["items"]:
    rows.append({
        "title": item["title"],
        "private_price": item["price"],
        "live_price": (item["live_match"] or {}).get("price"),
        "discrepancy": (item["discrepancy"] or {}).get("detail"),
    })
pd.DataFrame(rows)

## Summary

- **`rag.search`** (Shane): hybrid vector+BM25 retrieval over the private Amazon-2020 Household-Cleaning slice, with metadata filters and `doc_id` citations. Schema: `mcp/README_mcp_rag.md`.
- **`web.search`** (Clark): live multi-provider web search, `url` citations, degrades gracefully with no API key. Schema: `web-search-mcp/README_mcp_web.md`.
- **Combined MCP server** (Clark): both tools discoverable and callable over one `FastMCP` instance, timestamped request/response + source-URL logging. `web-search-mcp/combined_mcp_server.py`.
- **Reconciliation** (Clark): private fact always wins as the grounded baseline; conflicts are flagged, never silently overwritten. `src/rag/reconcile.py`.

Not covered by this notebook (rest of the team's parts): Router/Planner/Answerer-Critic LangGraph orchestration, ASR/TTS, and the Streamlit UI.